# ESM3 蛋白质结构预测

使用 ESM3 模型预测蛋白质三维结构。

**输入：** FASTA 格式的蛋白质序列文件  
**输出：** PDB 格式的结构文件

**系统要求：**
- GPU 推荐（可加速预测）
- 约 10-20 GB 磁盘空间

## 1. 初始化环境

In [ ]:
from protflow.utils.notebook_utils import init_notebook, ESM3_PACKAGES

# 自动初始化：检测环境、设置路径、安装依赖
paths = init_notebook('structure_prediction', packages=ESM3_PACKAGES)
WORK_DIR = paths['WORK_DIR']
DATA_DIR = paths['DATA_DIR']

## 2. 导入后端模块

In [ ]:
from protflow.prediction.esm3_predict import (
    predict_structures_from_fasta,
    ESM3GenerationConfig
)
from protflow.utils.seq_parser import extract_proteins_from_gbk
from pathlib import Path

## 3. 准备输入文件

支持以下输入格式：
- GenBank 文件 (.gbk, .gbff) - 自动提取蛋白质序列
- FASTA 文件 (.faa, .fa, .fasta) - 直接使用

In [ ]:
# 统一使用 INPUTS_DIR，每个功能有各自的子目录
# 蛋白质结构预测的输入文件应放在：INPUTS_DIR / 'protein_structure' / 

INPUTS_DIR = paths.get('INPUTS_DIR', DATA_DIR / 'inputs')
protein_input_dir = INPUTS_DIR / 'protein_structure'
protein_input_dir.mkdir(exist_ok=True, parents=True)

# 方式1: 从 GenBank 文件提取
gbk_dir = protein_input_dir / 'gbk_input'
gbk_dir.mkdir(exist_ok=True, parents=True)
output_fasta = WORK_DIR / 'proteins.faa'

if gbk_dir.exists() and list(gbk_dir.glob('*.gbk')):
    print("从 GenBank 文件提取蛋白质序列...")
    count = extract_proteins_from_gbk(gbk_dir, output_fasta)
    print(f"✓ 提取了 {count} 个蛋白质序列")
    input_file = output_fasta
else:
    # 方式2: 直接使用 FASTA 文件
    input_file = protein_input_dir / 'proteins.faa'
    if not input_file.exists():
        print(f"⚠️ 请设置输入文件: {input_file}")
        print(f"\n请将输入文件放在以下目录：")
        print(f"  GenBank文件: {gbk_dir}")
        print(f"  或FASTA文件: {protein_input_dir}")
    else:
        print(f"✓ 使用文件: {input_file}")

## 4. 配置 ESM3 参数

In [ ]:
# 配置 ESM3 生成参数
gen_config = ESM3GenerationConfig(
    track='structure',      # 'sequence', 'structure', 'function'
    num_steps=8,           # 生成步数（8-16，越大质量可能越好但更慢）
    temperature=None        # 温度参数（None 使用模型默认值）
)

print(f"ESM3 配置:")
print(f"  track: {gen_config.track}")
print(f"  num_steps: {gen_config.num_steps}")
print(f"  temperature: {gen_config.temperature}")

## 5. 运行结构预测

In [ ]:
# 输出目录
pdb_output_dir = WORK_DIR / 'predicted_structures'

# 运行预测（所有业务逻辑在后端）
if input_file.exists():
    results = predict_structures_from_fasta(
        fasta_file=input_file,
        out_dir=pdb_output_dir,
        generation_config=gen_config,
        min_seq_length=30,
        max_seq_length=2000,
        show_progress=True,
        skip_existing=True
    )
    
    print(f"\n✓ 预测完成！")
    print(f"  成功: {results['success']}")
    print(f"  跳过: {results['skipped']}")
    print(f"  错误: {results['errors']}")
    print(f"  过滤: {results['filtered']}")
    print(f"\n输出目录: {pdb_output_dir}")
else:
    print("⚠️ 未找到输入文件，请先设置正确的文件路径")

## 6. 查看结果

预测的 PDB 文件保存在输出目录中，可以直接用于后续分析。

In [ ]:
# 列出生成的 PDB 文件
if pdb_output_dir.exists():
    pdb_files = list(pdb_output_dir.glob('*.pdb'))
    print(f"共生成 {len(pdb_files)} 个 PDB 文件:")
    for pdb_file in pdb_files[:10]:  # 显示前10个
        print(f"  - {pdb_file.name}")
    if len(pdb_files) > 10:
        print(f"  ... 还有 {len(pdb_files) - 10} 个文件")

# 蛋白质结构预测工作流

**主要功能：**
- 从 `.gbk` 文件提取蛋白质CDS序列
- 使用 ESM3 进行蛋白质结构预测（支持所有官方参数）
- 生成标准PDB格式文件

**输入：**
- GenBank文件 (`.gbk`, `.gbff`)
- 或蛋白质序列文件 (`.faa`, `.fa`, `.fasta`)

**输出：**
- 预测的结构文件 (PDB格式)
- 结构预测报告

**系统要求：**
- JupyterLab/JupyterHub 服务器环境
- GPU 推荐用于加速结构预测
- 约 10-20 GB 磁盘空间

**更新说明：**
- ✅ 使用共享代码，消除重复
- ✅ 支持所有ESM3官方参数（temperature、track等）
- ✅ 自动依赖管理

## 1. 环境设置与依赖安装

In [ ]:
# 使用共享工具初始化环境（自动安装依赖、设置路径）
import sys
from pathlib import Path

# 添加protflow到路径
project_root = Path.cwd()
while not (project_root / 'src' / 'protflow').exists() and project_root != project_root.parent:
    project_root = project_root.parent

if (project_root / 'src').exists():
    src_dir = str(project_root / 'src')
    if src_dir not in sys.path:
        sys.path.insert(0, src_dir)
    print(f"✓ protflow 路径: {src_dir}")

# 导入并设置环境
from protflow.utils.notebook_utils import setup_esm3_notebook
from protflow.prediction.esm3_predict import load_esm3_small, predict_structures_from_fasta, ESM3GenerationConfig
from protflow.utils.seq_parser import extract_proteins_from_gbk

# 设置环境（自动检查和安装依赖）
paths = setup_esm3_notebook(work_dir_name='structure_prediction_runs')

PROJECT_ROOT = paths['PROJECT_ROOT']
WORK_DIR = paths['WORK_DIR']
DATA_DIR = paths['DATA_DIR']

print(f"\n✓ 环境初始化完成")
print(f"  工作目录: {WORK_DIR}")
print(f"  项目根目录: {PROJECT_ROOT}")

## 2. 文件输入

支持以下输入格式：
- GenBank文件 (.gbk, .gbff)
- 蛋白质序列文件 (.faa, .fa, .fasta)

In [ ]:
# 使用共享模块提取蛋白质序列
from Bio import SeqIO
import pandas as pd

# 方式1: 从GenBank文件提取（使用共享函数）
gbk_dir = DATA_DIR / 'inputs' / 'gbk_input'
output_fasta = WORK_DIR / 'proteins.faa'

if gbk_dir.exists() and (list(gbk_dir.glob('*.gbk')) or list(gbk_dir.glob('*.gbff'))):
    print("从GenBank文件提取蛋白质序列...")
    count = extract_proteins_from_gbk(gbk_dir, output_fasta)
    print(f"✓ 从GenBank文件提取了 {count} 个蛋白质序列")
    input_file = output_fasta
else:
    # 方式2: 直接使用FASTA文件
    input_file = DATA_DIR / 'inputs' / 'proteins.faa'  # 请替换为您的文件路径
    if not input_file.exists():
        print(f"⚠️ 请设置正确的输入文件路径: {input_file}")
        print("   支持格式: .gbk, .gbff, .faa, .fa, .fasta")
    else:
        print(f"✓ 使用FASTA文件: {input_file}")

# 读取序列用于后续处理
if input_file.exists():
    sequences = list(SeqIO.parse(input_file, 'fasta'))
    print(f"✓ 共读取 {len(sequences)} 个蛋白质序列")
else:
    sequences = []
    print("⚠️ 未找到输入文件，请先设置正确的文件路径")

## 3. 配置ESM3参数

使用`ESM3GenerationConfig`配置所有官方参数：

In [ ]:
# 配置ESM3参数（支持所有官方参数）
gen_config = ESM3GenerationConfig(
    track='structure',      # 'sequence', 'structure', 'function'
    num_steps=8,           # 生成步数（8-16，越大质量可能越好但更慢）
    temperature=None        # 温度参数（可选，None使用模型默认值）
)

print("ESM3配置:")
print(f"  track: {gen_config.track}")
print(f"  num_steps: {gen_config.num_steps}")
print(f"  temperature: {gen_config.temperature}")

## 4. 蛋白质结构预测

使用共享模块进行结构预测（支持所有ESM3官方参数）

In [ ]:
# 输出目录
pdb_output_dir = WORK_DIR / 'predicted_structures'

# 使用便捷函数一键预测（自动加载模型、处理序列、保存结果）
if input_file.exists() and len(sequences) > 0:
    print(f"\n开始预测 {len(sequences)} 个蛋白质的结构...")
    results = predict_structures_from_fasta(
        fasta_file=input_file,
        out_dir=pdb_output_dir,
        generation_config=gen_config,
        min_seq_length=30,   # 最小序列长度
        max_seq_length=2000, # 最大序列长度（可根据需要调整）
        show_progress=True,
        skip_existing=True
    )
    
    print(f"\n✓ 预测完成！")
    print(f"  成功: {results['success']}")
    print(f"  跳过: {results['skipped']}")
    print(f"  错误: {results['errors']}")
    print(f"  过滤: {results['filtered']}")
    
    # 转换为旧格式以兼容后续代码
    prediction_results = []
    if results['success'] > 0:
        for rec in sequences:
            pdb_file = pdb_output_dir / f"{rec.id.replace('|', '_').replace('/', '_')[:80]}.pdb"
            if pdb_file.exists():
                prediction_results.append({
                    'protein_id': rec.id,
                    'description': rec.description,
                    'sequence_length': len(rec.seq),
                    'pdb_file': str(pdb_file),
                    'status': 'success'
                })
else:
    print("⚠️ 无有效输入文件或序列，跳过预测")
    prediction_results = []
    results = {'success': 0, 'skipped': 0, 'errors': 0, 'filtered': 0}

## 5. 结果汇总与可视化

In [ ]:
# 生成结果报告
if len(prediction_results) > 0:
    results_df = pd.DataFrame(prediction_results)
    print("=== 结构预测结果汇总 ===")
    print(f"总蛋白质数: {len(results_df)}")
    print(f"预测成功: {len(results_df[results_df['status'] == 'success'])}")
    print(f"预测失败: {len(results_df[results_df['status'] != 'success'])}")
    
    # 保存结果
    results_file = WORK_DIR / 'prediction_summary.csv'
    results_df.to_csv(results_file, index=False)
    print(f"\n结果已保存: {results_file}")
    
    # 显示成功预测的统计信息
    if len(results_df[results_df['status'] == 'success']) > 0:
        successful = results_df[results_df['status'] == 'success']
        print(f"\n成功预测的平均序列长度: {successful['sequence_length'].mean():.1f}")
        print(f"序列长度范围: {successful['sequence_length'].min()} - {successful['sequence_length'].max()}")
        
        # 显示前几个成功的预测
        print("\n前5个成功预测:")
        print(successful[['protein_id', 'description', 'sequence_length']].head())
else:
    print("⚠️ 无预测结果")

## 6. 结构质量评估

（可选）对预测的结构进行质量评估

In [ ]:
# 使用后端模块进行结构质量评估（所有业务逻辑在后端）
from protflow.core.structure_analysis import assess_structure_quality
from pathlib import Path

# 评估预测的结构
if len(prediction_results) > 0:
    print("正在评估预测结构的质量...")
    quality_results = []

    for result in prediction_results:
        if result['status'] == 'success' and result['pdb_file']:
            quality = assess_structure_quality(Path(result['pdb_file']))
            # 转换为简单格式
            quality_results.append({
                'file': quality['file_path'],
                'residues': quality['num_residues'],
                'atoms': quality['num_atoms'],
                'status': quality['status']
            })

    if quality_results:
        quality_df = pd.DataFrame(quality_results)
        print(f"评估完成，平均残基数: {quality_df['residues'].mean():.1f}")
        print(f"评估完成，平均原子数: {quality_df['atoms'].mean():.1f}")
        
        # 保存质量评估结果
        quality_file = WORK_DIR / 'structure_quality.csv'
        quality_df.to_csv(quality_file, index=False)
        print(f"质量评估结果保存: {quality_file}")
else:
    print("⚠️ 无预测结果，跳过质量评估")

## 7. 下一步操作

完成结构预测后，您可以：

1. **口袋检测**: 使用 `02_pocket_detection_p2rank.ipynb` 进行结合口袋预测
2. **结构比对**: 使用 `12_structure_alignment_dali.ipynb` 进行结构相似性分析
3. **分子对接**: 使用 `03_ligand_docking_vina.ipynb` 进行配体对接（需要先进行口袋检测）

**结果文件位置:**
- 预测结构: `{WORK_DIR}/predicted_structures/`
- 结果汇总: `{WORK_DIR}/prediction_summary.csv`
- 质量评估: `{WORK_DIR}/structure_quality.csv`